# One-vs-Rest LightGBM
        **tabular › multilabel** · notebook `ovr_lightgbm`

        **Kapan dipakai:** Hubungan non-linear per label.

        **Jebakan umum:** 14 model terpisah; label sangat langka butuh penanganan khusus.

**Data demo:** Yeast (2417 gen, 103 fitur, 14 label fungsi protein). Satu baris boleh punya banyak label sekaligus (kolom 0/1).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular_extra/multilabel/yeast_train.csv"; TEST_PATH = ROOT / "data_latihan/tabular_extra/multilabel/yeast_test.csv"; ID_COL = 'id'
SEED, N_SPLITS, VALID_SIZE = 42, 3, 0.2
MODEL_KEY = "ovr_lightgbm"; KIND = "tree"
OUT = out_dir("tabular", "multilabel", MODEL_KEY); set_seed(SEED)

## 2. Data
Kolom label = kolom yang ada di train tetapi tidak ada di test.

In [ ]:
from superdb.tabular import detect_columns, prepare_features, make_preprocessor
df = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
TARGETS = [c for c in df.columns if c not in test.columns]; Y = df[TARGETS].to_numpy(int)
cols = detect_columns(df, target=None, id_col=ID_COL, drop_cols=TARGETS, verbose=False); X, NUM, CAT = prepare_features(df, cols)
print(X.shape, "| label:", len(TARGETS), "| rata-rata label per baris:", round(Y.sum(1).mean(), 2), "| baris tanpa label:", int((Y.sum(1) == 0).sum()))
display(pd.Series(Y.mean(0), index=TARGETS).round(3).to_frame("proporsi positif").T)

## 3. Pemisahan dan model
Prapemrosesan (imputasi, skala/encoding) berada di dalam Pipeline.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.base import clone
Xtr, Xva, Ytr, Yva = train_test_split(X, Y, test_size=VALID_SIZE, random_state=SEED)

import numpy as np
from sklearn.multiclass import OneVsRestClassifier
from lightgbm import LGBMClassifier

clf = OneVsRestClassifier(LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=10, colsample_bytree=0.7, random_state=42, n_jobs=1, verbose=-1))
pipe = Pipeline([("pre", make_preprocessor(NUM, CAT, kind=KIND)), ("clf", clf)])
pipe

## 4. Cross-validation

In [ ]:
cvr = cross_validate(pipe, Xtr, Ytr, cv=KFold(N_SPLITS, shuffle=True, random_state=SEED), scoring=["f1_micro", "f1_macro"])
print({k: round(float(v.mean()), 4) for k, v in cvr.items() if k.startswith("test_")})

## 5. Evaluasi
**f1_micro** (didominasi label sering), **f1_macro** (tiap label setara), **hamming loss**, dan **subset accuracy** (semua label benar sekaligus).

In [ ]:
from sklearn.metrics import f1_score, hamming_loss, accuracy_score
def scores(m, A):
    if hasattr(m, "predict_proba"):
        P = m.predict_proba(A); return np.column_stack([q[:, 1] for q in P]) if isinstance(P, list) else np.asarray(P)
    return m.decision_function(A)
pipe.fit(Xtr, Ytr); S = scores(pipe, Xva); P = pipe.predict(Xva)
metrics = {"f1_micro": f1_score(Yva, P, average="micro", zero_division=0), "f1_macro": f1_score(Yva, P, average="macro", zero_division=0), "hamming_loss": hamming_loss(Yva, P), "subset_accuracy": accuracy_score(Yva, P)}
display(show_table(metrics)); display(pd.DataFrame({"f1": f1_score(Yva, P, average=None, zero_division=0), "support": Yva.sum(0)}, index=TARGETS).T.round(3))

## 6. Threshold per label (dicari dari OOF pada training)
Label langka butuh ambang berbeda dari 0.5.

In [ ]:
oof = np.zeros(Ytr.shape, float)
for a, b in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(Xtr):
    oof[b] = scores(clone(pipe).fit(Xtr.iloc[a], Ytr[a]), Xtr.iloc[b])
THR = []
for j in range(len(TARGETS)):
    cand = np.quantile(oof[:, j], np.linspace(0.05, 0.95, 60))
    THR.append(cand[int(np.argmax([f1_score(Ytr[:, j], oof[:, j] >= t, zero_division=0) for t in cand]))])
P2 = (S >= np.array(THR)).astype(int)
print(f"f1_micro: {metrics['f1_micro']:.4f} -> {f1_score(Yva, P2, average='micro', zero_division=0):.4f} | f1_macro: {metrics['f1_macro']:.4f} -> {f1_score(Yva, P2, average='macro', zero_division=0):.4f}")
metrics["f1_micro_thr"] = f1_score(Yva, P2, average="micro", zero_division=0); metrics["f1_macro_thr"] = f1_score(Yva, P2, average="macro", zero_division=0)

## 7. Latih ulang pada semua data → submission → simpan

In [ ]:
final = clone(pipe).fit(X, Y)
Xte, _, _ = prepare_features(test, cols); Xte = Xte.reindex(columns=X.columns)
Pt = (scores(final, Xte) >= np.array(THR)).astype(int)
sub = pd.DataFrame(Pt, columns=TARGETS); sub.insert(0, ID_COL, test[ID_COL].to_numpy()); sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
save_model({"pipe": final, "thresholds": THR, "targets": TARGETS, "cols": cols}, OUT / "model.joblib")
save_json({"cv": {k: float(v.mean()) for k, v in cvr.items() if k.startswith("test_")}, "valid": metrics, "thresholds": dict(zip(TARGETS, map(float, THR)))}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib"); assert (b["pipe"].predict(Xte.head(30)) == final.predict(Xte.head(30))).all(); print("OK. hasil di", OUT)

## Langkah berikutnya
- Model native multi-output (Random Forest/ExtraTrees/CatBoost MultiLogloss) menangkap korelasi antar-label; bandingkan dengan One-vs-Rest.
- Format submission lomba bisa 'daftar label dipisah spasi': ubah dari matriks 0/1 di sel terakhir.